In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import plotly.graph_objects as go


In [ ]:
energy_path = "/content/drive/MyDrive/household_power_consumption.txt"

df = pd.read_csv(
    energy_path,
    sep=";",
    parse_dates={"datetime": ["Date", "Time"]},
    na_values="?",
    low_memory=False
)

df["Global_active_power"] = pd.to_numeric(df["Global_active_power"], errors="coerce")
df.dropna(inplace=True)

df["date"] = df["datetime"].dt.date


/tmp/ipython-input-3123396122.py:3: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(
/tmp/ipython-input-3123396122.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(


In [ ]:
daily_energy = (
    df.groupby("date")["Global_active_power"]
    .mean()
    .reset_index()
)

daily_energy["Energy_kWh"] = daily_energy["Global_active_power"] * 24
daily_energy.drop(columns=["Global_active_power"], inplace=True)

daily_energy["date"] = pd.to_datetime(daily_energy["date"])
daily_energy.head()


,date,Energy_kWh
0,2006-12-16,73.283394
1,2006-12-17,56.507667
2,2006-12-18,36.730433
3,2006-12-19,27.769900
4,2006-12-20,37.095800


In [ ]:
exam_path = "/content/drive/MyDrive/exam_calendar.csv"

exam_df = pd.read_csv(
    exam_path,
    parse_dates=["start_date", "end_date"]
)

exam_df


,event,start_date,end_date
0,Mid-Semester Exams,2023-10-09,2023-10-21
1,End-Semester Exams,2023-11-27,2023-12-16
2,Mid-Semester Exams,2024-03-04,2024-03-16
3,End-Semester Exams,2024-04-22,2024-05-11
4,Mid-Semester Exams,2024-09-09,2024-09-21
5,End-Semester Exams,2024-11-25,2024-12-14


In [ ]:
daily_energy["is_exam_period"] = 0

for _, row in exam_df.iterrows():
    daily_energy.loc[
        (daily_energy["date"] >= row["start_date"]) &
        (daily_energy["date"] <= row["end_date"]),
        "is_exam_period"
    ] = 1

daily_energy[daily_energy["is_exam_period"] == 1].head()


,date,Energy_kWh,is_exam_period


In [ ]:
ts = daily_energy.set_index("date")["Energy_kWh"]

model = ExponentialSmoothing(
    ts,
    trend="add",
    seasonal="add",
    seasonal_periods=7
)

fit = model.fit()

forecast_days = 30
forecast = fit.forecast(forecast_days)

forecast_value = round(forecast.mean(), 2)
forecast_value


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


np.float64(27.57)

In [ ]:
fig = go.Figure(
    go.Indicator(
        mode="gauge+number",
        value=forecast_value,
        title={"text": "Forecasted Library Energy Usage During Exams (kWh/day)"},
        gauge={
            "axis": {"range": [0, ts.max() * 1.2]},
            "steps": [
                {"range": [0, ts.mean()], "color": "lightgreen"},
                {"range": [ts.mean(), ts.mean() * 1.2], "color": "yellow"},
                {"range": [ts.mean() * 1.2, ts.max() * 1.2], "color": "red"},
            ],
            "threshold": {
                "line": {"color": "black", "width": 4},
                "value": forecast_value
            }
        }
    )
)

fig.show()


In [ ]:
import plotly.graph_objects as go
import numpy as np

# Define max for gauge
max_val = int(np.ceil(ts.max() * 1.2))

# Tick values every 5 units
tick_vals = list(range(0, max_val + 1, 5))

# Tick labels (numbers by default)
tick_text = [str(v) for v in tick_vals]

# Replace first and last with Min/Max labels
tick_text[0] = f"Minimum\n(0)"
tick_text[-1] = f"Maximum\n({max_val})"

fig = go.Figure()

fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=forecast_value,
        number={"font": {"size": 64}},
        gauge={
            "axis": {
                "range": [0, max_val],
                "tickvals": tick_vals,
                "ticktext": tick_text
            },
            "steps": [
                {"range": [0, ts.mean()], "color": "lightgreen"},
                {"range": [ts.mean(), ts.mean() * 1.2], "color": "yellow"},
                {"range": [ts.mean() * 1.2, max_val], "color": "red"},
            ],
            "threshold": {
                "line": {"color": "black", "width": 4},
                "value": forecast_value
            },
            "bar": {"color": "darkgreen"},
        },
        domain={"x": [0, 1], "y": [0.35, 1]}
    )
)

# Title at top
fig.update_layout(
    title={
        "text": "Forecasted Library Energy Usage During Exams (kWh/day)",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.95
    },
    margin=dict(t=80, b=80)
)

# Explanation text below the number
fig.add_annotation(
    x=0.5,
    y=0.18,
    xref="paper",
    yref="paper",
    text=f"During exams, the library is expected to consume about {forecast_value} units of electricity per day.",
    showarrow=False,
    font=dict(size=16),
    align="center"
)

fig.show()
